# Lab 03 — Reading Execution Plans

**Tujuan:** Membedakan parsed, optimized, dan physical plan serta pushdown/penyederhanaan.

**Lingkungan:** Spark standalone (`spark-master:7077`), Jupyter, dua worker. Jalankan sel secara berurutan. Setiap notebook dapat dijalankan sendiri.

**Metode:** Prediksi → jalankan → amati `explain()`/Spark UI → ubah satu variabel → simpulkan.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Spark Training - Lab 03")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/10 00:41:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.7
Master: spark://spark-master:7077


## 1. Read the plan

In [2]:
df = spark.range(0, 1_000_000, 1, 4)
step1 = df.filter("id > 1000")
step2 = step1.filter("id < 2000")
step3 = step2.select("id")
step3.explain(True)

== Parsed Logical Plan ==
'Project ['id]
+- Filter (id#0L < cast(2000 as bigint))
   +- Filter (id#0L > cast(1000 as bigint))
      +- Range (0, 1000000, step=1, splits=Some(4))

== Analyzed Logical Plan ==
id: bigint
Project [id#0L]
+- Filter (id#0L < cast(2000 as bigint))
   +- Filter (id#0L > cast(1000 as bigint))
      +- Range (0, 1000000, step=1, splits=Some(4))

== Optimized Logical Plan ==
Filter ((id#0L > 1000) AND (id#0L < 2000))
+- Range (0, 1000000, step=1, splits=Some(4))

== Physical Plan ==
*(1) Filter ((id#0L > 1000) AND (id#0L < 2000))
+- *(1) Range (0, 1000000, step=1, splits=4)



Cari bagian **Optimized Logical Plan**. Apakah Spark mempertahankan semua langkah persis sebagaimana ditulis?

In [3]:
step3.explain("formatted")
step3.count()

== Physical Plan ==
* Filter (2)
+- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 1000000, step=1, splits=Some(4))

(2) Filter [codegen id : 1]
Input [1]: [id#0L]
Condition : ((id#0L > 1000) AND (id#0L < 2000))




999

## 2. Add a wide transformation

Cari `Exchange` dan `HashAggregate`.

In [8]:
groups = df.withColumn("group_id", F.col("id") % 100)
summary = groups.groupBy("group_id").count()
summary.explain("formatted")
summary.show(5)

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Project (2)
            +- Range (1)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 1000000, step=1, splits=Some(4))

(2) Project
Output [1]: [(id#0L % 100) AS group_id#94L]
Input [1]: [id#0L]

(3) HashAggregate
Input [1]: [group_id#94L]
Keys [1]: [group_id#94L]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#103L]
Results [2]: [group_id#94L, count#104L]

(4) Exchange
Input [2]: [group_id#94L, count#104L]
Arguments: hashpartitioning(group_id#94L, 8), ENSURE_REQUIREMENTS, [plan_id=274]

(5) HashAggregate
Input [2]: [group_id#94L, count#104L]
Keys [1]: [group_id#94L]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#99L]
Results [2]: [group_id#94L, count(1)#99L AS count#100L]

(6) AdaptiveSparkPlan
Output [2]: [group_id#94L, count#100L]
Arguments: isFinalPlan=false


+--------+-----+
|group_id|count|
+--------+-----+
|       2|10000

**Tugas:** bandingkan plan filter vs plan groupBy; identifikasi komponen yang bisa menyebabkan stage boundary.

## Refleksi peserta

Tuliskan: **(1)** apa prediksi Anda, **(2)** apa yang teramati di Spark UI atau output, **(3)** mengapa Spark berperilaku demikian. Hindari menyimpulkan dari waktu eksekusi satu kali saja.

In [10]:
spark.stop()